# D20C–D20E — Estimação causal principal, robustezes e auditoria

## tl;dr

Na especificação principal congelada, o ATT simple foi **2.050,99 empregos assalariados**, com banda simultânea de 95% **[1.481,32; 2.620,66]**. O event-study principal foi `k=0: 818,28`, `k=1: 1.641,59` e `k=2: 2.188,81`; `k=-2` foi `-962,94`, com banda que não inclui zero. As sensibilidades em nível preservaram o sinal do ATT simple, mas os leads negativos, o timing proxy, os cinco tratados extremos, a influência de porte e a coorte 2013 com `n=2` permanecem riscos de interpretação. **Estes resultados não aprovam automaticamente o desenho causal.**

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display, Markdown

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from visualizacao_ipt import CORES_IPT, aplicar_tema_ipt, estilizar_tabela_ipt

OUT = ROOT / 'outputs' / 'causal'
FIG = ROOT / 'outputs' / 'figures' / 'interactive'

def mostrar(frame, n=30):
    display(estilizar_tabela_ipt(frame.head(n)))

def ler_csv(nome):
    return pd.read_csv(OUT / nome)

manifesto = json.loads((OUT / 'D20C_manifesto_resultado.json').read_text(encoding='utf-8'))
gates_d20d = json.loads((OUT / 'D20D_gates.json').read_text(encoding='utf-8'))
auditoria = json.loads((OUT / 'D20E_auditoria_pos_estimacao.json').read_text(encoding='utf-8'))
att_gt = ler_csv('D20C_att_gt_principal.csv')
simples = ler_csv('D20C_agregacao_simples.csv')
coorte = ler_csv('D20C_agregacao_coorte.csv')
evento = ler_csv('D20C_event_study_completo.csv')
evento_principal = ler_csv('D20C_event_study_principal.csv')
comparativa = ler_csv('D20D_tabela_comparativa.csv')
estabilidade = ler_csv('D20D_estabilidade.csv')
matriz = ler_csv('D20E_matriz_evidencia.csv')


## 1. Objetivo

Estimar, pela primeira vez no outcome real, o efeito da chegada operacional de campi Fase II sobre `pessoal_ocupado_assalariado` municipal, executar apenas as robustezes pré-especificadas e auditar os resultados sem criar novas especificações.

## 2. O que foi congelado antes dos resultados

Unidade município-ano; período 2007–2019; tratamento pela primeira presença operacional; outcome CEMPRE 708 em nível; never-treated; `anticipation=0`; `base_period='universal'`; `est_method='dr'`; painel verdadeiro; referência `k=-1`; 1.999 draws; seed 20260924; `n_jobs=1`; bandas simultâneas de 95%; cluster por entidade.

In [2]:
mostrar(pd.DataFrame({'parametro': manifesto['configuracao'].keys(), 'valor': [str(v) for v in manifesto['configuracao'].values()]}), 30)

,parametro,valor
0,backend,differences==0.3.0
1,outcome,pessoal_ocupado_assalariado
2,base_period,universal
3,control_group,never_treated
4,est_method,dr
5,anticipation,0
6,as_repeated_cross_section,False
7,boot_iterations,1999
8,random_state,20260924
9,n_jobs,1


## 3. População documental versus estimável

A D15 documental contém 129 tratados. A view D20B principal contém 128: Cabo Frio/RJ permanece documentado, mas é excluído integralmente da view porque `g-1=2009` é um ano de transição institucional inelegível. O código `5003900` continua ausente por sigilo.

In [3]:
pop = manifesto['populacao']
mostrar(pd.DataFrame([{'tratados_documentais_D15': 129, 'tratados_estimaveis': pop['n_tratados'], 'controles': pop['n_controles'], 'municipios': pop['n_municipios'], 'linhas': pop['n_linhas'], 'painel_balanceado': pop['painel_balanceado']}]))

,tratados_documentais_D15,tratados_estimaveis,controles,municipios,linhas,painel_balanceado
0,129,128,4963,5091,66183,True


## 4. Estimador

O backend `differences==0.3.0` estima ATT grupo-tempo compatível com Callaway–Sant'Anna. `ATT(g,t)` responde ao efeito médio no ano `t` para a coorte tratada em `g`; `simple` resume as células elegíveis; `cohort` resume cada coorte; `event` agrega por tempo relativo. Esses objetos respondem a perguntas distintas e não são intercambiáveis.

## 5. ATT(g,t)

Foram estimadas 65 células grupo-tempo, incluindo 45 pós-tratamento. Nenhuma célula pós-tratamento falhou. A tabela completa permanece no artefato congelado; abaixo aparece uma amostra auditável.

In [4]:
mostrar(att_gt[['coorte_g', 'ano', 'event_time', 'periodo_base', 'att', 'std_error', 'ci_lower', 'ci_upper', 'n_tratados_relevante', 'status_celula']], 25)

,coorte_g,ano,event_time,periodo_base,att,std_error,ci_lower,ci_upper,n_tratados_relevante,status_celula
0,2009,2007,-2,2008,-303.197461,319.175011,-1146.709121,540.314198,21,ESTIMADA
1,2009,2008,-1,2008,0.000000,nan,nan,nan,21,REFERENCIA_K_MENOS_1
2,2009,2009,0,2008,1122.233461,390.109323,91.257505,2153.209417,21,ESTIMADA
3,2009,2010,1,2008,2485.495668,597.032544,907.665656,4063.325680,21,ESTIMADA
4,2009,2011,2,2008,3519.706293,870.834403,1218.276212,5821.136375,21,ESTIMADA
5,2009,2012,3,2008,4219.046276,958.451403,1686.063101,6752.029451,21,ESTIMADA
6,2009,2013,4,2008,4938.435355,1074.137257,2099.719079,7777.151631,21,ESTIMADA
7,2009,2014,5,2008,5367.329304,964.260777,2818.993189,7915.665418,21,ESTIMADA
8,2009,2015,6,2008,4618.658473,823.567684,2442.144274,6795.172671,21,ESTIMADA
9,2009,2016,7,2008,3889.667002,794.902517,1788.908744,5990.425261,21,ESTIMADA


## 6. Inferência

As bandas são simultâneas de 95%. O procedimento validado em D20B remove `k=-1` apenas do multiplier bootstrap e o recoloca como zero na apresentação, sem erro-padrão ou limites. A repetição com a mesma seed reproduziu todos os objetos com diferença máxima de `1,82e-12`, abaixo da tolerância numérica `1e-10`.

In [5]:
mostrar(pd.DataFrame([auditoria['reprodutibilidade_seed']]).T.reset_index().rename(columns={'index': 'item', 0: 'valor'}), 20)

,item,valor
0,reproduzivel,True
1,random_state,20260924
2,boot_iterations,1999
3,n_jobs,1
4,max_abs_diff,0.000000
5,tolerancia_numerica,0.000000
6,max_abs_diff_por_objeto,"{'att_gt': 1.8189894035458565e-12, 'simple': 0.0, 'cohort': 4.547473508864641e-13, 'event': 4.547473508864641e-13}"
7,duracao_repeticao_segundos,9.084402
8,outputs_principais_sobrescritos,False


## 7. Resultado principal

O ATT simple é uma estimativa agregada em nível de empregos assalariados. Significância estatística não elimina os riscos de identificação registrados antes e depois da estimação.

In [6]:
mostrar(simples[['att', 'std_error', 'ci_lower', 'ci_upper', 'inferencia']])

,att,std_error,ci_lower,ci_upper,inferencia
0,2050.990297,296.592364,1481.324482,2620.656112,bootstrap_simultanea_95


## 8. ATT por coorte

A coorte 2013 permanece com `n=2`. Sua magnitude negativa deve ser lida junto da grande incerteza; não é ausência de efeito nem base para exclusão retrospectiva.

In [7]:
mostrar(coorte[['coorte_g', 'n_tratados', 'att', 'std_error', 'ci_lower', 'ci_upper', 'suporte_temporal', 'warning_precisao']])
fig_coorte = go.Figure(go.Scatter(x=coorte['coorte_g'], y=coorte['att'], mode='markers', marker={'size': 11, 'color': CORES_IPT['AZUL_PRINCIPAL']}, error_y={'type': 'data', 'symmetric': False, 'array': coorte['ci_upper']-coorte['att'], 'arrayminus': coorte['att']-coorte['ci_lower']}))
fig_coorte.add_hline(y=0, line_dash='dash', line_color=CORES_IPT['AZUL_ESCURO'])
fig_coorte.update_xaxes(title='Coorte', dtick=1)
fig_coorte.update_yaxes(title='ATT agregado da coorte')
aplicar_tema_ipt(fig_coorte, titulo='D20C — ATT por coorte com bandas simultâneas de 95%')
fig_coorte.show()

,coorte_g,n_tratados,att,std_error,ci_lower,ci_upper,suporte_temporal,warning_precisao
0,2009,21,4048.598715,762.833075,2334.504518,5762.692913,2007-2019; k=-2...10,nan
1,2010,26,1694.989875,463.671607,653.114742,2736.865008,2007-2019; k=-3...9,nan
2,2011,66,1478.535552,421.758248,530.840194,2426.230911,2007-2019; k=-4...8,nan
3,2012,13,2345.844106,941.637746,229.973873,4461.714339,2007-2019; k=-5...7,nan
4,2013,2,-2200.017789,1640.189613,-5885.541647,1485.506070,2007-2019; k=-6...6,COORTE_2013_N2_PRECISAO_CRITICA


## 9. Event-study

A janela principal destaca `k=-2,-1,0,+1,+2`, mas o suporte completo foi estimado e preservado.

In [8]:
mostrar(evento_principal[['event_time', 'att', 'std_error', 'ci_lower', 'ci_upper', 'indicador_inferencia']])
fig_event = go.Figure(go.Scatter(x=evento_principal['event_time'], y=evento_principal['att'], mode='lines+markers', line={'color': CORES_IPT['AZUL_PRINCIPAL']}, error_y={'type': 'data', 'symmetric': False, 'array': evento_principal['ci_upper']-evento_principal['att'], 'arrayminus': evento_principal['att']-evento_principal['ci_lower']}))
fig_event.add_hline(y=0, line_dash='dash', line_color=CORES_IPT['AZUL_ESCURO'])
fig_event.add_vline(x=-1, line_dash='dot', line_color=CORES_IPT['CIANO'])
fig_event.update_xaxes(title='Tempo relativo ao tratamento (k)', dtick=1)
fig_event.update_yaxes(title='ATT — pessoal ocupado assalariado')
aplicar_tema_ipt(fig_event, titulo='D20C — Event-study principal')
fig_event.show()

,event_time,att,std_error,ci_lower,ci_upper,indicador_inferencia
0,-2,-962.935546,190.549055,-1457.761430,-468.109663,BANDA_SIMULTANEA_95
1,-1,0.000000,nan,nan,nan,REFERENCIA_DETERMINISTICA_SEM_INFERENCIA
2,0,818.277606,164.453550,391.217639,1245.337572,BANDA_SIMULTANEA_95
3,1,1641.588847,275.406220,926.402269,2356.775424,BANDA_SIMULTANEA_95
4,2,2188.811932,362.169275,1248.315367,3129.308496,BANDA_SIMULTANEA_95


## 10. Leads

Os leads disponíveis são diagnósticos pós-estimação. Três bandas simultâneas pré-tratamento excluem zero (`k=-4,-3,-2`). Isso não deve ser convertido mecanicamente em aprovação ou reprovação, mas aumenta a cautela diante da heterogeneidade pré já documentada em D17/D18 e não autoriza escolher outra janela depois de observar os resultados.

In [9]:
leads = evento.loc[evento['event_time'] < -1, ['event_time', 'att', 'std_error', 'ci_lower', 'ci_upper']]
mostrar(leads, 20)

,event_time,att,std_error,ci_lower,ci_upper
0,-6,-8867.934213,4001.420527,-19258.993092,1523.124666
1,-5,-6443.040218,2512.012326,-12966.340583,80.260147
2,-4,-3034.627062,551.138965,-4465.848148,-1603.405977
3,-3,-1877.943697,307.001652,-2675.178635,-1080.708759
4,-2,-962.935546,190.549055,-1457.761430,-468.109663


## 11. Robustez log1p

Com a mesma população, o ATT simple em log1p foi `0,00397`, com banda `[-0,02565; 0,03358]`. O estimando está em escala logarítmica e não pode ser comparado 1:1 a empregos em nível.

## 12. Janela com 3 pré

Restrita às coortes 2010–2013 (107 tratados), mantendo o painel integral e 4.963 controles, a agregação simple foi `1.576,25`, banda `[1.028,24; 2.124,26]`.

## 13. Sensibilidade de suporte

Excluindo somente os cinco tratados identificados no D18, restaram 123 tratados. O ATT simple foi `1.818,34`, banda `[1.289,50; 2.347,17]`. Esta é uma sensibilidade; não corrige nem substitui a amostra principal.

## 14. Spillover 25 km

Com 4.589 controles remanescentes, o ATT simple foi `2.069,87`, banda `[1.501,54; 2.638,19]`.

## 15. Spillover 50 km

Com 3.488 controles remanescentes, o ATT simple foi `2.070,07`, banda `[1.492,80; 2.647,35]`.

## 16. Spillover 100 km

Com 1.323 controles remanescentes, o ATT simple foi `2.096,00`, banda `[1.534,17; 2.657,84]`. Distância não prova spillover.

## 17. Arranjo populacional

Com 4.818 controles remanescentes, o ATT simple foi `2.080,00`, banda `[1.511,72; 2.648,28]`. A classificação IBGE é de 2010 e não representa integração contemporânea em todos os anos.

## 18. Comparação de sensibilidades

A tabela é factual: não há ranking, vencedor ou seleção por p-valor. Diferenças absolutas não são calculadas para log1p porque as escalas não são comparáveis.

In [10]:
mostrar(comparativa, 20)
mostrar(estabilidade, 20)

,especificacao,n_tratados,n_controles,escala_outcome,coortes,ATT_simple,banda_lower,banda_upper,observacao_metodologica,ATT_event_k0,ATT_event_k1,ATT_event_k2
0,principal,128,4963,nivel,"2009,2010,2011,2012,2013",2050.990297,1481.324482,2620.656112,Especificação principal congelada,818.277606,1641.588847,2188.811932
1,log1p,128,4963,log1p,"2009,2010,2011,2012,2013",0.003965,-0.025647,0.033577,Estimando em escala log1p; não comparável 1:1 a empregos em nível,-0.013101,0.014559,0.002766
2,janela_3pre,107,4963,nivel,"2010,2011,2012,2013",1576.250024,1028.235371,2124.264677,Coortes 2010-2013; painel 2007-2019 integral,758.622718,1475.962274,1927.608365
3,suporte,123,4963,nivel,"2009,2010,2011,2012,2013",1818.335515,1289.500480,2347.170550,Exclui apenas cinco tratados extremos D18; não é amostra principal,610.924231,1296.685982,1753.449137
4,spillover_25km,128,4589,nivel,"2009,2010,2011,2012,2013",2069.867704,1501.543226,2638.192181,Exclui controles pré-classificados a <=25 km,826.518987,1656.332730,2208.620230
5,spillover_50km,128,3488,nivel,"2009,2010,2011,2012,2013",2070.073768,1492.800360,2647.347176,Exclui controles pré-classificados a <=50 km,838.343958,1676.647842,2227.835241
6,spillover_100km,128,1323,nivel,"2009,2010,2011,2012,2013",2096.003193,1534.165318,2657.841068,Exclui controles pré-classificados a <=100 km,861.964811,1713.700840,2264.959768
7,arranjo_populacional,128,4818,nivel,"2009,2010,2011,2012,2013",2080.002383,1511.719797,2648.284968,Classificação IBGE 2010; sensibilidade temporalmente limitada,838.291736,1680.358060,2236.672170


,especificacao,escala_outcome,ATT_simple,banda_lower,banda_upper,diferenca_absoluta_principal,diferenca_relativa_principal,mudanca_sinal,intervalos_sobrepostos_descritivo
0,principal,nivel,2050.990297,1481.324482,2620.656112,0.000000,0.000000,False,True
1,log1p,log1p,0.003965,-0.025647,0.033577,nan,nan,nan,nan
2,janela_3pre,nivel,1576.250024,1028.235371,2124.264677,-474.740272,-0.231469,False,True
3,suporte,nivel,1818.335515,1289.500480,2347.170550,-232.654781,-0.113435,False,True
4,spillover_25km,nivel,2069.867704,1501.543226,2638.192181,18.877407,0.009204,False,True
5,spillover_50km,nivel,2070.073768,1492.800360,2647.347176,19.083472,0.009305,False,True
6,spillover_100km,nivel,2096.003193,1534.165318,2657.841068,45.012896,0.021947,False,True
7,arranjo_populacional,nivel,2080.002383,1511.719797,2648.284968,29.012086,0.014145,False,True


## 19. Auditoria pós-estimação

A auditoria confirmou células pós-tratamento finitas, ausência de duplicidade, event-time correto, referência `k=-1`, coortes, hashes e reprodutibilidade. A saída pública do backend não expõe pesos de agregação; as tabelas `simple`, `cohort` e `event` são as agregações oficiais da API, sem ponderação alternativa inferida. A contextualização `ATT simple / média pré dos tratados = 11,93%` é apenas ordem de grandeza descritiva, não novo estimando causal.

In [11]:
mostrar(pd.DataFrame({'verificacao': auditoria['verificacoes'].keys(), 'valor': auditoria['verificacoes'].values()}), 20)
mostrar(matriz, 20)

,verificacao,valor
0,att_gt_sem_duplicidade,True
1,att_gt_pos_finito,True
2,event_time_correto,True
3,referencia_k_menos_1,True
4,coortes_completas,True
5,simple_unico,True
6,hash_d15_integro,True
7,hash_view_d20b_integro,True


,dimensao,evidencia_observada,resultado_principal,sensibilidades,risco_residual,impacto_interpretacao
0,pre_tendencias,5 leads; máximo |ATT|=8867.934213177514,Leads reportados com bandas simultâneas,Janela 3 pré preservada,Poder limitado e heterogeneidade pré,Diagnóstico; não prova tendências paralelas
1,overlap,Cinco tratados extremos diagnosticados antes da estimação,Principal preserva 128 tratados,Suporte: ATT simple=1818.3355151088697,Sensibilidade não corrige o principal,Comparar magnitude sem substituir estimando
2,escala,Diferenças de nível e assimetria documentadas em D17/D18,Outcome em nível,log1p em escala distinta,Não há conversão exata para empregos,Não comparar ATT log 1:1 com nível
3,spillover,Controles próximos/mesmo arranjo pré-classificados,Sem filtro espacial,ATT simple variou entre 2069.8677039876516 e 2096.0031930755117,Distância e arranjo são proxies,"Diferenças são compatíveis, não prova de spillover"
4,timing_proxy,128/129 timings D15 eram proxy Censo; Cabo Frio tratado por addendum,anticipation=0,Nenhuma janela alternativa executada,Antecipação substantiva permanece possível,"Hipótese identificadora, não fato observado"
5,coorte_pequena,Coorte 2013 n=2; ATT=-2200.017788779828,Coorte preservada,Presente em sensibilidades aplicáveis,Alta imprecisão e baixa generalização,Separar magnitude de incerteza
6,influencia_municipios_grandes,D18 mostrou influência material pré-tratamento,Nenhuma exclusão retrospectiva,Não foi criada robustez top-N,Estimativas em nível podem ser influenciadas por porte,Interpretar como efeito médio em nível
7,inferencia,"Multiplier bootstrap, 1.999 draws, cluster entidade",Bandas simultâneas 95%,Mesma configuração em todas robustezes,Poucas unidades na coorte 2013,Incerteza deve acompanhar cada estimativa
8,validade_externa,População principal efetivamente estimável: 128 tratados,Estimando restrito ao suporte observado,Sensibilidades mudam população/controles,Não representa automaticamente os 147 municípios institucionais,Generalização deve permanecer delimitada


## 20. Limitações

Os leads negativos são materialmente relevantes; cinco tratados estão além do máximo do score dos controles; o outcome em nível é influenciado por porte; o timing é majoritariamente proxy; spillovers são plausíveis e não observados diretamente; a coorte 2013 tem apenas dois municípios; o CEMPRE não cobre toda atividade econômica municipal; e a validade externa não alcança automaticamente os 147 municípios institucionais. Resultados significativos não removem essas ressalvas, e resultados não significativos não equivalem a efeito zero.

## 21. Gates

Os gates técnicos D20C, D20D e D20E foram concluídos. `DESENHO_CAUSAL_APROVADO = NAO` e `D20E_INTERPRETACAO_CAUSAL_PENDENTE_REVISAO = SIM`. A próxima decisão é humana; D21 não foi executada.

In [12]:
gates = {**manifesto['gates'], **gates_d20d, **auditoria['gates']}
mostrar(pd.DataFrame({'gate': gates.keys(), 'valor': gates.values()}), 40)

,gate,valor
0,D20C_AMOSTRA_PRINCIPAL_VALIDADA,SIM
1,D20C_ATT_GT_REAL_ESTIMADO,SIM
2,D20C_AGREGACAO_SIMPLE_ESTIMADA,SIM
3,D20C_AGREGACAO_COORTE_ESTIMADA,SIM
4,D20C_EVENT_STUDY_REAL_ESTIMADO,SIM
5,D20C_BOOTSTRAP_1999_CONCLUIDO,SIM
6,D20C_BANDAS_SIMULTANEAS_VALIDAS,SIM
7,D20C_RESULTADO_PRINCIPAL_CONGELADO,SIM
8,DESENHO_CAUSAL_APROVADO,NAO
9,D20D_LOG1P_EXECUTADO,SIM
